In [3]:
# mypy: allow-untyped-defs
from typing import Optional, Union

import torch
from torch import Tensor
from torch.distributions import constraints
from torch.distributions.distribution import Distribution
from torch.distributions.utils import (
    broadcast_all,
    lazy_property,
    logits_to_probs,
    probs_to_logits,
)
from torch.distributions import Normal
from torch.fft import fft
import math
import torch.nn.functional as F  # Added for softplus

__all__ = ["PoissonBinomial"]


def _clamp_by_zero(x):
    # works like clamp(x, min=0) but has grad at 0 is 0.5
    return (x.clamp(min=0) + x - x.clamp(max=0)) / 2


class PoissonBinomial(Distribution):
    """
    Creates a Poisson Binomial distribution parameterized by either :attr:`probs` or :attr:`logits` (but not both),
    where each Bernoulli trial can have a different probability of success. The :attr:`probs` or :attr:`logits`
    must have an additional trailing dimension corresponding to the number of trials.

    This implementation supports multiple computation methods:
    - 'dp': Dynamic programming (exact, O(n^2))
    - 'dft': Discrete Fourier Transform using FFT (exact, O(n log n))
    - 'normal': Normal approximation
    - 'rna': Refined Normal Approximation (with skewness correction for quantiles)
    - 'auto': Automatically selects based on n (dp for n<=256, dft for 256<n<=1024, rna for n>1024)

    Example::

        >>> # Single distribution with 3 trials
        >>> probs = torch.tensor([0.1, 0.2, 0.3])
        >>> m = PoissonBinomial(probs=probs, method='dft')
        >>> x = m.sample((5,))  # 5 samples

        >>> # Batched distribution with batch size 2, each with 2 trials using approximation
        >>> probs = torch.tensor([[0.5, 0.5], [0.2, 0.8]])
        >>> m = PoissonBinomial(probs=probs, method='normal')
        >>> x = m.sample()

    Args:
        probs (Tensor): Event probabilities with shape (*, n_trials)
        logits (Tensor): Event log-odds with shape (*, n_trials)
        method (str): Computation method ('auto', 'dp', 'dft', 'normal', 'rna')
        validate_args (bool): Whether to validate arguments
    """

    arg_constraints = {
        "probs": constraints.unit_interval,
        "logits": constraints.real,
    }
    has_enumerate_support = True

    def __init__(
        self,
        probs: Optional[Tensor] = None,
        logits: Optional[Tensor] = None,
        method: str = "auto",
        validate_args: Optional[bool] = None,
    ) -> None:
        if (probs is None) == (logits is None):
            raise ValueError(
                "Either `probs` or `logits` must be specified, but not both."
            )
        if method not in ["auto", "dp", "dft", "normal", "rna"]:
            raise ValueError(f"Invalid method: {method}")
        self.method = method
        if probs is not None:
            self.probs = probs
        else:
            assert logits is not None
            self.logits = logits

        self._param = self.probs if probs is not None else self.logits
        if len(self._param.shape) < 1:
            raise ValueError("The parameter must have at least one dimension for trials.")
        self._num_trials = self._param.shape[-1]
        batch_shape = self._param.shape[:-1]
        super().__init__(batch_shape, validate_args=validate_args)

    def expand(self, batch_shape, _instance=None):
        new = self._get_checked_instance(PoissonBinomial, _instance)
        batch_shape = torch.Size(batch_shape)
        if "probs" in self.__dict__:
            new.probs = self.probs.expand(batch_shape + (self._num_trials,))
            new._param = new.probs
        if "logits" in self.__dict__:
            new.logits = self.logits.expand(batch_shape + (self._num_trials,))
            new._param = new.logits
        super(PoissonBinomial, new).__init__(batch_shape, validate_args=False)
        new._validate_args = self._validate_args
        new._num_trials = self._num_trials
        new.method = self.method
        return new

    def _new(self, *args, **kwargs):
        return self._param.new(*args, **kwargs)

    @constraints.dependent_property(is_discrete=True, event_dim=0)
    def support(self):
        return constraints.integer_interval(0, self._num_trials)

    @property
    def mean(self) -> Tensor:
        return self.probs.sum(-1)

    @property
    def mode(self) -> Tensor:
        if self._is_exact_method():
            return self._log_pmf().exp().argmax(-1)
        else:
            return self.mean.round().clamp(0, self._num_trials).long()

    @property
    def variance(self) -> Tensor:
        return (self.probs * (1 - self.probs)).sum(-1)

    @property
    def skewness(self) -> Tensor:
        p = self.probs
        term = p * (1 - p) * (1 - 2 * p)
        return term.sum(-1) / self.variance.clamp(min=1e-8).pow(1.5)

    @lazy_property
    def logits(self) -> Tensor:
        return probs_to_logits(self.probs, is_binary=True)

    @lazy_property
    def probs(self) -> Tensor:
        return logits_to_probs(self.logits, is_binary=True)

    @property
    def param_shape(self) -> torch.Size:
        return self._param.size()

    def _is_exact_method(self):
        m = self.method
        if m == "auto":
            return self._num_trials <= 1024
        return m in ["dp", "dft"]

    def _select_method(self):
        m = self.method
        if m == "auto":
            if self._num_trials <= 256:
                return "dp"
            elif self._num_trials <= 1024:
                return "dft"
            else:
                return "rna"
        return m

    def _pmf_dft(self):
        p = self.probs  # batch_shape x n_trials
        n = self._num_trials
        N = n + 1
        device = p.device
        real_dtype = p.dtype
        complex_dtype = torch.complex128 if real_dtype == torch.float64 else torch.complex64
        j = torch.arange(N, device=device, dtype=real_dtype)
        omega = torch.exp(2j * math.pi * j / N)  # exp(+2 pi i j / N)
        cf = torch.ones(self.batch_shape + (N,), dtype=complex_dtype, device=device)
        for i in range(n):
            term = (1 - p[..., i].unsqueeze(-1)) + p[..., i].unsqueeze(-1) * omega
            cf = cf * term
        # P(k) = 1/N * fft(cf)[k].real
        pmf = fft(cf, dim=-1).real / N
        return pmf.clamp(min=0)  # Clamp for numerical errors

    def _log_pmf_dp(self):
        logits = self.logits  # batch_shape x num_trials
        n = self._num_trials
        log_dp = logits.new_full(self.batch_shape + (n + 1,), float("-inf"))
        log_dp[..., 0] = 0.0
        for t in range(n):
            log_p = -F.softplus(-logits[..., t])
            log_q = -F.softplus(logits[..., t])
            new_log_dp = log_dp.new_zeros(log_dp.shape)
            new_log_dp[..., 0] = log_dp[..., 0] + log_q
            log_dp_k = log_dp[..., 1:] + log_q.unsqueeze(-1)
            log_dp_km1 = log_dp[..., :-1] + log_p.unsqueeze(-1)
            new_log_dp[..., 1:] = torch.logsumexp(torch.stack((log_dp_k, log_dp_km1)), dim=0)
            log_dp = new_log_dp
        return log_dp

    def _log_pmf(self):
        sel_method = self._select_method()
        if sel_method == "dp":
            return self._log_pmf_dp()
        elif sel_method == "dft":
            pmf = self._pmf_dft()
            return torch.log(pmf + 1e-30)
        else:
            raise RuntimeError("log_pmf not available for approximation methods")

    def sample(self, sample_shape=torch.Size()):
        shape = self._extended_shape(sample_shape)
        with torch.no_grad():
            return torch.bernoulli(self.probs.expand(shape + (self._num_trials,))).sum(-1)

    def log_prob(self, value):
        if self._validate_args:
            self._validate_sample(value)
        sel_method = self._select_method()
        if sel_method in ["dp", "dft"]:
            log_dp = self._log_pmf()  # batch_shape x (num_trials + 1)
            value = value.long()
            sample_shape = value.shape[: len(value.shape) - len(self.batch_shape)]
            log_dp_expanded = log_dp.expand(sample_shape + log_dp.shape)
            value = value.clamp(min=0, max=self._num_trials).unsqueeze(-1)
            return log_dp_expanded.gather(-1, value).squeeze(-1)
        else:
            std = self.variance.clamp(min=1e-8).sqrt()
            normal = Normal(self.mean, std)
            upper = normal.cdf(value + 0.5)
            lower = normal.cdf(value - 0.5)
            return torch.log((upper - lower).clamp(min=1e-10))

    def cdf(self, value):
        if self._validate_args:
            self._validate_sample(value)
        sel_method = self._select_method()
        if sel_method in ["dp", "dft"]:
            log_pmf = self._log_pmf()
            pmf = torch.exp(log_pmf)
            cdf = torch.cumsum(pmf, dim=-1)
            value = value.long()
            sample_shape = value.shape[: len(value.shape) - len(self.batch_shape)]
            cdf_expanded = cdf.expand(sample_shape + cdf.shape)
            value = value.clamp(min=0, max=self._num_trials).unsqueeze(-1)
            return cdf_expanded.gather(-1, value).squeeze(-1)
        else:
            std = self.variance.clamp(min=1e-8).sqrt()
            normal = Normal(self.mean, std)
            return normal.cdf(value + 0.5)

    def icdf(self, value):
        if not constraints.unit_interval.check(value).all():
            raise ValueError("icdf input must be in [0,1]")
        sel_method = self._select_method()
        if sel_method in ["dp", "dft"]:
            log_pmf = self._log_pmf()
            pmf = torch.exp(log_pmf)
            cdf = torch.cumsum(pmf, dim=-1)
            sample_shape = value.shape[: len(value.shape) - len(self.batch_shape)]
            value = value.unsqueeze(-1)
            cdf_expanded = cdf.expand(sample_shape + cdf.shape)
            lt = cdf_expanded < value
            k = lt.sum(-1)
            return k.clamp(max=self._num_trials)
        else:
            std = self.variance.clamp(min=1e-8).sqrt()
            normal_std = Normal(0, 1)
            z = normal_std.icdf(value)
            if sel_method == "rna":
                gamma = self.skewness
                z = z + (z.pow(2) - 1) * gamma / 6
            q = self.mean + std * z
            return q.round().clamp(0, self._num_trials)

    def entropy(self):
        sel_method = self._select_method()
        if sel_method in ["dp", "dft"]:
            log_dp = self._log_pmf()
            pmf = torch.exp(log_dp)
            return -(pmf * log_dp).where(log_dp > float("-inf"), torch.zeros_like(log_dp)).sum(-1)
        else:
            std = self.variance.clamp(min=1e-8).sqrt()
            normal = Normal(self.mean, std)
            return normal.entropy()

    def enumerate_support(self, expand=True):
        n = self._num_trials
        values = torch.arange(
            n + 1, dtype=self._param.dtype, device=self._param.device
        )
        values = values.view((-1,) + (1,) * len(self.batch_shape))
        if expand:
            values = values.expand((-1,) + self.batch_shape)
        return values

In [4]:
probs = torch.tensor([0.1, 0.2, 0.3])  # n=3
m = PoissonBinomial(probs=probs)  # auto -> dp
print(m.mean)  # tensor(0.6000)
print(m.variance)  # tensor(0.4600)
print(m.skewness)  # tensor(0.3674)  # Manual: sum p(1-p)(1-2p) = 0.1*0.9*(-0.8)+... ≈0.078, / (0.46)^{1.5} ≈0.078/0.212≈0.367
print(m.mode)  # tensor(0)  # As before
print(m.sample((2,)))  # e.g., tensor([1., 0.])
print(m.log_prob(torch.tensor(1)))  # ≈ tensor(-0.9217)  # log(0.398)
print(m.cdf(torch.tensor(1)))  # ≈ tensor(0.902)  # P(0)+P(1)=0.504+0.398=0.902
print(m.icdf(torch.tensor(0.5)))  # tensor(1)  # Since cdf(0)=0.504 >=0.5? Wait, smallest k >=, but cdf(0)=0.504, for u=0.5 -> k=0 if cdf(0)>=0.5, but wait: sum(lt)=0 if value<=cdf[0], but 0.5 <0.504? No, if u=0.5, lt = [0.5 < cdf[0]=0.504] False, sum=0, but actually since cdf[0]=P(0), icdf finds where cum >=u, so if u<=P(0), k=0
# For u=0.6, icdf=1 since cdf(0)=0.504 <0.6, sum lt=1
print(m.entropy())  # ≈ tensor(1.1410)

tensor(0.6000)
tensor(0.4600)
tensor(0.8077)
tensor(0)
tensor([2., 0.])
tensor(-0.9213)
tensor(0.9020)
tensor(0)
tensor(0.9622)


In [6]:
probs = torch.tensor([0.1]*10 + [0.9]*10)  # n=20, to test dft
m = PoissonBinomial(probs=probs, method='dft')
print(m.mean)  # tensor(10.0000)  # 10*0.1 +10*0.9=1+9=10
print(m.variance)  # tensor(2.0000)  # 10*0.1*0.9 +10*0.9*0.1=10*0.09*2=1.8? Wait 20*0.09=1.8? No: 10*0.09 +10*0.09=1.8
print(m.skewness)  # tensor(0.0000)  # Symmetric
print(m.mode)  # tensor(10)
print(m.sample())  # e.g., tensor(10.)
print(m.log_prob(torch.tensor(10)))  # log PMF at 10, exact via DFT
print(m.cdf(torch.tensor(10)))  # cum up to 10
print(m.icdf(torch.tensor(0.5)))  # ≈ tensor(10)
# DFT should match DP for small n, but useful for larger.

tensor(10.)
tensor(1.8000)
tensor(-6.1704e-08)
tensor(10)
tensor(8.)
tensor(-1.1628)
tensor(0.6563)
tensor(10)


In [7]:
probs = torch.tensor([0.5] * 2000)  # n=2000, auto -> rna since >1024
m = PoissonBinomial(probs=probs)
print(m.mean)  # tensor(1000.)
print(m.variance)  # tensor(500.)  # 2000*0.5*0.5=500
print(m.skewness)  # tensor(0.0000)  # Since all p=0.5, (1-2p)=0
print(m.mode)  # tensor(1000)  # round mean
print(m.sample())  # tensor(1001.)  # etc., exact sample
print(m.log_prob(torch.tensor(1000)))  # Approx log(normal pdf-ish), ≈ log(1/sqrt(2 pi var)) ≈ -log(sqrt(2 pi 500)) ≈ -6.05
print(m.cdf(torch.tensor(1000)))  # ≈0.5  # normal.cdf(mean +0.5)
print(m.icdf(torch.tensor(0.5)))  # tensor(1000)  # Since gamma=0, same as normal
print(m.entropy())  # ≈ tensor(6.592)  # normal entropy = 0.5 log(2 pi e var) ≈ 0.5 log(2 pi e 500) ≈6.592

tensor(1000.)
tensor(500.)
tensor(0.)
tensor(1000)
tensor(1006.)
tensor(-4.0263)
tensor(0.5089)
tensor(1000.)
tensor(4.5262)


In [9]:
probs = torch.tensor([[0.1]*100, [0.9]*100])  # batch 2, n=100 each
m = PoissonBinomial(probs=probs, method='normal')
print(m.mean)  # tensor([10., 90.])
print(m.variance)  # tensor([9., 9.])  # 100*0.1*0.9=9
print(m.skewness)  # tensor([ 0.6667, -0.6667])  # (1-2p)>0 for p=0.1, negative for 0.9
print(m.log_prob(torch.tensor([10, 90])))  # Approx log P
print(m.cdf(torch.tensor([10, 90])))  # ≈ tensor([0.5, 0.5])
print(m.icdf(torch.tensor([0.95, 0.95])))  # Approx quantile, for normal ≈ mu + 1.645 sigma ≈10+1.645*3=14.9→15, 90+4.9→95

tensor([10.0000, 90.0000])
tensor([9.0000, 9.0000])
tensor([ 0.2667, -0.2667])
tensor([-2.0222, -2.0222])
tensor([0.5662, 0.5662])
tensor([15., 95.])


In [10]:
probs = torch.tensor([0.1]*100)  # n=100, mean=10, var=9, gamma≈0.6667 (positive skew)
m = PoissonBinomial(probs=probs, method='rna')
print(m.icdf(torch.tensor(0.95)))  # Adjusted: z=1.645, adj=1.645 + (1.645^2 -1)*0.6667/6 ≈1.645 + (2.706-1)*0.1111≈1.645+0.19≈1.835
# q=10 + 3*1.835≈10+5.5≈15.5→16 (higher than normal 15 due to positive skew)
# Other methods same as normal, but icdf adjusted.

tensor(15.)


# Version 2

In [1]:
# mypy: allow-untyped-defs
from typing import Optional, Tuple

import torch
from torch import Tensor
from torch.distributions import constraints
from torch.distributions.distribution import Distribution
from torch.distributions.utils import (
    lazy_property,
    logits_to_probs,
    probs_to_logits,
)
from torch.distributions import Normal
from torch.fft import fft, ifft
import math
import torch.nn.functional as F

__all__ = ["PoissonBinomial"]


def _clamp_by_zero(x):
    # works like clamp(x, min=0) but has grad at 0 is 0.5
    return (x.clamp(min=0) + x - x.clamp(max=0)) / 2


class PoissonBinomial(Distribution):
    """
    Poisson Binomial distribution: sum of independent Bernoulli trials with heterogeneous probabilities.

    Methods:
    - Exact:
      * 'dp'  : Dynamic programming in log-space (O(n^2)) — Chen & Liu (1997)
      * 'dft' : Discrete Fourier Transform via FFT (O(n log n)) — Hong (2013)
    - Approximate:
      * 'normal' : Normal approximation with continuity correction
      * 'rna'    : Refined Normal Approximation (Cornish–Fisher skewness correction) — Volkova (2005)
      * 'saddle' : Saddlepoint approximation (Lugannani–Rice) for PMF/CDF and tails

    Auto selection:
      - n <= 256   : 'dp'
      - 256 < n <= 2048 : 'dft'
      - n > 2048   : 'saddle' (tails) or 'rna' (bulk), chosen adaptively

    Reference formulas used:
      - DFT/FFT characteristic function: Hong (2013)
      - DP recursion: Chen & Liu (1997)
      - Moments (mean, variance, skewness): Fernández & Williams (2010)
      - Cornish–Fisher skewness correction: Volkova (2005)
      - Saddlepoint (Lugannani–Rice) approximation: standard form for discrete sums of Bernoulli RVs
    """

    arg_constraints = {
        "probs": constraints.unit_interval,
        "logits": constraints.real,
    }
    has_enumerate_support = True

    def __init__(
        self,
        probs: Optional[Tensor] = None,
        logits: Optional[Tensor] = None,
        method: str = "auto",
        validate_args: Optional[bool] = None,
        cache: bool = True,
    ) -> None:
        if (probs is None) == (logits is None):
            raise ValueError("Either `probs` or `logits` must be specified, but not both.")
        if method not in ["auto", "dp", "dft", "normal", "rna", "saddle"]:
            raise ValueError(f"Invalid method: {method}")
        self.method = method
        self._cache_enabled = cache
        self._cached_pmf = None
        self._cached_cdf = None

        if probs is not None:
            self.probs = probs
        else:
            assert logits is not None
            self.logits = logits

        self._param = self.probs if probs is not None else self.logits
        if len(self._param.shape) < 1:
            raise ValueError("The parameter must have at least one dimension for trials.")
        self._num_trials = self._param.shape[-1]
        batch_shape = self._param.shape[:-1]
        super().__init__(batch_shape, validate_args=validate_args)

    def expand(self, batch_shape, _instance=None):
        new = self._get_checked_instance(PoissonBinomial, _instance)
        batch_shape = torch.Size(batch_shape)
        if "probs" in self.__dict__:
            new.probs = self.probs.expand(batch_shape + (self._num_trials,))
            new._param = new.probs
        if "logits" in self.__dict__:
            new.logits = self.logits.expand(batch_shape + (self._num_trials,))
            new._param = new.logits
        super(PoissonBinomial, new).__init__(batch_shape, validate_args=False)
        new._validate_args = self._validate_args
        new._num_trials = self._num_trials
        new.method = self.method
        new._cache_enabled = self._cache_enabled
        new._cached_pmf = None
        new._cached_cdf = None
        return new

    def _new(self, *args, **kwargs):
        return self._param.new(*args, **kwargs)

    @constraints.dependent_property(is_discrete=True, event_dim=0)
    def support(self):
        return constraints.integer_interval(0, self._num_trials)

    @property
    def mean(self) -> Tensor:
        return self.probs.sum(-1)

    @property
    def mode(self) -> Tensor:
        if self._is_exact_method():
            return self._log_pmf().exp().argmax(-1)
        else:
            return self.mean.round().clamp(0, self._num_trials).long()

    @property
    def variance(self) -> Tensor:
        return (self.probs * (1 - self.probs)).sum(-1)

    @property
    def skewness(self) -> Tensor:
        p = self.probs
        term = p * (1 - p) * (1 - 2 * p)
        return term.sum(-1) / self.variance.clamp(min=1e-8).pow(1.5)

    @lazy_property
    def logits(self) -> Tensor:
        return probs_to_logits(self.probs, is_binary=True)

    @lazy_property
    def probs(self) -> Tensor:
        return logits_to_probs(self.logits, is_binary=True)

    @property
    def param_shape(self) -> torch.Size:
        return self._param.size()

    def _is_exact_method(self):
        m = self.method
        if m == "auto":
            return self._num_trials <= 2048
        return m in ["dp", "dft"]

    def _select_method(self, value: Optional[Tensor] = None):
        m = self.method
        if m != "auto":
            return m
        n = self._num_trials
        if n <= 256:
            return "dp"
        elif n <= 2048:
            return "dft"
        else:
            # For large n, use saddlepoint for tails and RNA for bulk
            if value is not None:
                # Heuristic: tails if far from mean by > 2 std
                std = self.variance.clamp(min=1e-8).sqrt()
                dev = (value - self.mean).abs()
                return "saddle" if (dev > 2 * std).any() else "rna"
            return "saddle"

    # ---------------------------
    # Exact PMF via DFT (Hong, 2013)
    # ---------------------------
    def _pmf_dft(self):
        p = self.probs  # batch_shape x n_trials
        n = self._num_trials
        N = n + 1
        device = p.device
        real_dtype = p.dtype
        complex_dtype = torch.complex128 if real_dtype == torch.float64 else torch.complex64
        j = torch.arange(N, device=device, dtype=real_dtype)
        omega = torch.exp(2j * math.pi * j / N)  # exp(+2 pi i j / N)
        cf = torch.ones(self.batch_shape + (N,), dtype=complex_dtype, device=device)
        for i in range(n):
            term = (1 - p[..., i].unsqueeze(-1)) + p[..., i].unsqueeze(-1) * omega
            cf = cf * term
        # PMF via inverse DFT: P(k) = (1/N) * Re{ifft(cf)[k]}
        pmf = ifft(cf, dim=-1).real / 1.0  # ifft already includes 1/N scaling in PyTorch
        pmf = pmf.clamp(min=0)  # Clamp for numerical errors
        return pmf

    # ---------------------------
    # Exact PMF via DP (Chen & Liu, 1997)
    # ---------------------------
    def _log_pmf_dp(self):
        logits = self.logits  # batch_shape x num_trials
        n = self._num_trials
        log_dp = logits.new_full(self.batch_shape + (n + 1,), float("-inf"))
        log_dp[..., 0] = 0.0
        for t in range(n):
            log_p = -F.softplus(-logits[..., t])
            log_q = -F.softplus(logits[..., t])
            new_log_dp = log_dp.new_zeros(log_dp.shape)
            new_log_dp[..., 0] = log_dp[..., 0] + log_q
            log_dp_k = log_dp[..., 1:] + log_q.unsqueeze(-1)
            log_dp_km1 = log_dp[..., :-1] + log_p.unsqueeze(-1)
            new_log_dp[..., 1:] = torch.logsumexp(torch.stack((log_dp_k, log_dp_km1)), dim=0)
            log_dp = new_log_dp
        return log_dp

    def _log_pmf(self):
        if self._cache_enabled and self._cached_pmf is not None:
            return self._cached_pmf
        sel_method = self._select_method()
        if sel_method == "dp":
            log_pmf = self._log_pmf_dp()
        elif sel_method == "dft":
            pmf = self._pmf_dft()
            log_pmf = torch.log(pmf + 1e-30)
        else:
            raise RuntimeError("log_pmf not available for approximation methods")
        if self._cache_enabled:
            self._cached_pmf = log_pmf
        return log_pmf

    def _cdf_exact(self):
        if self._cache_enabled and self._cached_cdf is not None:
            return self._cached_cdf
        log_pmf = self._log_pmf()
        pmf = torch.exp(log_pmf)
        cdf = torch.cumsum(pmf, dim=-1)
        if self._cache_enabled:
            self._cached_cdf = cdf
        return cdf

    # ---------------------------
    # Saddlepoint (Lugannani–Rice) approximation
    # ---------------------------
    def _cgf(self, t: Tensor) -> Tensor:
        # K(t) = sum log(1 - p_i + p_i e^t)
        p = self.probs
        et = torch.exp(t)
        return torch.sum(torch.log1p(p * (et - 1)), dim=-1)

    def _cgf_prime(self, t: Tensor) -> Tensor:
        # K'(t) = sum p_i e^t / (1 - p_i + p_i e^t)
        p = self.probs
        et = torch.exp(t)
        return torch.sum(p * et / (1 - p + p * et), dim=-1)

    def _cgf_double_prime(self, t: Tensor) -> Tensor:
        # K''(t) = sum p_i(1-p_i) e^t / (1 - p_i + p_i e^t)^2
        p = self.probs
        et = torch.exp(t)
        denom = (1 - p + p * et)
        return torch.sum(p * (1 - p) * et / (denom * denom), dim=-1)

    def _solve_saddlepoint(self, k: Tensor, max_iter: int = 50, tol: float = 1e-10) -> Tensor:
        # Solve K'(t) = k for t via Newton's method
        # Shape: broadcast to batch_shape
        t = torch.zeros_like(self.mean)  # initial guess
        for _ in range(max_iter):
            Kp = self._cgf_prime(t)
            Kpp = self._cgf_double_prime(t).clamp(min=1e-12)
            step = (Kp - k) / Kpp
            t = t - step
            if torch.max(torch.abs(step)) < tol:
                break
        return t

    def _pmf_saddle(self, k: Tensor) -> Tensor:
        # Lugannani–Rice PMF approximation for discrete lattice:
        # P(Y=k) ≈ (1 / sqrt(2π K''(t))) * exp(K(t) - k t)
        # where t solves K'(t) = k
        t = self._solve_saddlepoint(k)
        K = self._cgf(t)
        Kpp = self._cgf_double_prime(t).clamp(min=1e-12)
        pmf = torch.exp(K - k * t) / torch.sqrt(2 * math.pi * Kpp)
        return pmf.clamp(min=0)

    def _cdf_saddle(self, k: Tensor) -> Tensor:
        # Lugannani–Rice CDF approximation:
        # F(k) ≈ Φ(w) + φ(w) * (1/w - 1/u)
        # where w = sign(t) * sqrt(2(k t - K(t))), u = t * sqrt(K''(t))
        t = self._solve_saddlepoint(k)
        K = self._cgf(t)
        Kpp = self._cgf_double_prime(t).clamp(min=1e-12)
        w = torch.sign(t) * torch.sqrt(torch.clamp(2 * (k * t - K), min=0))
        u = t * torch.sqrt(Kpp)
        normal0 = Normal(0.0, 1.0)
        Phi_w = normal0.cdf(w)
        phi_w = torch.exp(normal0.log_prob(w))
        # Handle w≈0 safely
        inv_w = torch.where(torch.abs(w) > 1e-12, 1.0 / w, torch.zeros_like(w))
        cdf = Phi_w + phi_w * (inv_w - 1.0 / u.clamp(min=1e-12))
        return cdf.clamp(0.0, 1.0)

    # ---------------------------
    # RNA / Normal approximations
    # ---------------------------
    def _normal_interval_prob(self, value: Tensor) -> Tensor:
        std = self.variance.clamp(min=1e-8).sqrt()
        normal = Normal(self.mean, std)
        upper = normal.cdf(value + 0.5)
        lower = normal.cdf(value - 0.5)
        return (upper - lower).clamp(min=1e-12)

    def _rna_interval_prob(self, value: Tensor) -> Tensor:
        # Apply Cornish–Fisher skewness correction to continuity-corrected bounds
        std = self.variance.clamp(min=1e-8).sqrt()
        mu = self.mean
        gamma = self.skewness
        normal0 = Normal(0.0, 1.0)

        def cf_correct(z: Tensor) -> Tensor:
            return z + (z.pow(2) - 1.0) * gamma / 6.0

        # Transform bounds to z, correct, then back
        z_upper = (value + 0.5 - mu) / std
        z_lower = (value - 0.5 - mu) / std
        z_upper_cf = cf_correct(z_upper)
        z_lower_cf = cf_correct(z_lower)
        # Approximate interval probability via corrected z-bounds
        return (normal0.cdf(z_upper_cf) - normal0.cdf(z_lower_cf)).clamp(min=1e-12)

    # ---------------------------
    # Public API
    # ---------------------------
    def sample(self, sample_shape=torch.Size()):
        shape = self._extended_shape(sample_shape)
        with torch.no_grad():
            return torch.bernoulli(self.probs.expand(shape + (self._num_trials,))).sum(-1)

    def log_prob(self, value):
        if self._validate_args:
            self._validate_sample(value)
        sel_method = self._select_method(value)
        if sel_method in ["dp", "dft"]:
            log_dp = self._log_pmf()  # batch_shape x (num_trials + 1)
            value = value.long()
            sample_shape = value.shape[: len(value.shape) - len(self.batch_shape)]
            log_dp_expanded = log_dp.expand(sample_shape + log_dp.shape)
            value = value.clamp(min=0, max=self._num_trials).unsqueeze(-1)
            return log_dp_expanded.gather(-1, value).squeeze(-1)
        elif sel_method == "normal":
            return torch.log(self._normal_interval_prob(value))
        elif sel_method == "rna":
            return torch.log(self._rna_interval_prob(value))
        elif sel_method == "saddle":
            # PMF via saddlepoint
            k = value.clamp(min=0, max=self._num_trials).to(self.mean.dtype)
            pmf = self._pmf_saddle(k)
            return torch.log(pmf.clamp(min=1e-30))
        else:
            raise RuntimeError(f"Unknown method: {sel_method}")

    def cdf(self, value):
        if self._validate_args:
            self._validate_sample(value)
        sel_method = self._select_method(value)
        if sel_method in ["dp", "dft"]:
            cdf = self._cdf_exact()
            value = value.long()
            sample_shape = value.shape[: len(value.shape) - len(self.batch_shape)]
            cdf_expanded = cdf.expand(sample_shape + cdf.shape)
            value = value.clamp(min=0, max=self._num_trials).unsqueeze(-1)
            return cdf_expanded.gather(-1, value).squeeze(-1)
        elif sel_method == "normal":
            std = self.variance.clamp(min=1e-8).sqrt()
            normal = Normal(self.mean, std)
            return normal.cdf(value + 0.5)
        elif sel_method == "rna":
            # Use CF-corrected z for CDF at continuity-corrected bound
            std = self.variance.clamp(min=1e-8).sqrt()
            mu = self.mean
            gamma = self.skewness
            normal0 = Normal(0.0, 1.0)
            z = (value + 0.5 - mu) / std
            z_cf = z + (z.pow(2) - 1.0) * gamma / 6.0
            return normal0.cdf(z_cf)
        elif sel_method == "saddle":
            k = value.clamp(min=0, max=self._num_trials).to(self.mean.dtype)
            return self._cdf_saddle(k)
        else:
            raise RuntimeError(f"Unknown method: {sel_method}")

    def icdf(self, value, max_iter: int = 30):
        if not constraints.unit_interval.check(value).all():
            raise ValueError("icdf input must be in [0,1]")
        sel_method = self._select_method()
        if sel_method in ["dp", "dft"]:
            cdf = self._cdf_exact()
            sample_shape = value.shape[: len(value.shape) - len(self.batch_shape)]
            value = value.unsqueeze(-1)
            cdf_expanded = cdf.expand(sample_shape + cdf.shape)
            lt = cdf_expanded < value
            k = lt.sum(-1)
            return k.clamp(max=self._num_trials)
        else:
            # Newton refinement on approximate CDF to solve F(k) = u
            # Initialize with RNA quantile
            std = self.variance.clamp(min=1e-8).sqrt()
            mu = self.mean
            normal0 = Normal(0.0, 1.0)
            z0 = normal0.icdf(value)
            gamma = self.skewness
            if sel_method == "rna":
                z0 = z0 + (z0.pow(2) - 1.0) * gamma / 6.0
            q = (mu + std * z0).round().clamp(0, self._num_trials).to(self.mean.dtype)

            def approx_cdf(k):
                if sel_method == "normal":
                    std_ = std
                    normal_ = Normal(mu, std_)
                    return normal_.cdf(k + 0.5)
                elif sel_method == "rna":
                    z = (k + 0.5 - mu) / std
                    z_cf = z + (z.pow(2) - 1.0) * gamma / 6.0
                    return normal0.cdf(z_cf)
                elif sel_method == "saddle":
                    return self._cdf_saddle(k)
                else:
                    return self._cdf_saddle(k)

            def approx_pmf(k):
                if sel_method == "normal":
                    return self._normal_interval_prob(k)
                elif sel_method == "rna":
                    return self._rna_interval_prob(k)
                elif sel_method == "saddle":
                    return self._pmf_saddle(k)
                else:
                    return self._pmf_saddle(k)

            # Newton iterations on discrete lattice using derivative ~ pmf(k)
            for _ in range(max_iter):
                Fk = approx_cdf(q)
                diff = Fk - value
                pmf_q = approx_pmf(q).clamp(min=1e-12)
                step = diff / pmf_q
                q_new = (q - step).round().clamp(0, self._num_trials)
                if torch.max(torch.abs(q_new - q)) < 1e-6:
                    break
                q = q_new
            return q

    def entropy(self):
        sel_method = self._select_method()
        if sel_method in ["dp", "dft"]:
            log_dp = self._log_pmf()
            pmf = torch.exp(log_dp)
            return -(pmf * log_dp).where(log_dp > float("-inf"), torch.zeros_like(log_dp)).sum(-1)
        else:
            # Approximate entropy via normal
            std = self.variance.clamp(min=1e-8).sqrt()
            normal = Normal(self.mean, std)
            return normal.entropy()

    def enumerate_support(self, expand=True):
        n = self._num_trials
        values = torch.arange(n + 1, dtype=self._param.dtype, device=self._param.device)
        values = values.view((-1,) + (1,) * len(self.batch_shape))
        if expand:
            values = values.expand((-1,) + self.batch_shape)
        return values


c:\Users\matth\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
probs = torch.tensor([0.1, 0.2, 0.3])  # n=3
m = PoissonBinomial(probs=probs)  # auto -> dp
print(m.mean)  # tensor(0.6000)
print(m.variance)  # tensor(0.4600)
print(m.skewness)  # tensor(0.3674)  # Manual: sum p(1-p)(1-2p) = 0.1*0.9*(-0.8)+... ≈0.078, / (0.46)^{1.5} ≈0.078/0.212≈0.367
print(m.mode)  # tensor(0)  # As before
print(m.sample((2,)))  # e.g., tensor([1., 0.])
print(m.log_prob(torch.tensor(1)))  # ≈ tensor(-0.9217)  # log(0.398)
print(m.cdf(torch.tensor(1)))  # ≈ tensor(0.902)  # P(0)+P(1)=0.504+0.398=0.902
print(m.icdf(torch.tensor(0.5)))  # tensor(1)  # Since cdf(0)=0.504 >=0.5? Wait, smallest k >=, but cdf(0)=0.504, for u=0.5 -> k=0 if cdf(0)>=0.5, but wait: sum(lt)=0 if value<=cdf[0], but 0.5 <0.504? No, if u=0.5, lt = [0.5 < cdf[0]=0.504] False, sum=0, but actually since cdf[0]=P(0), icdf finds where cum >=u, so if u<=P(0), k=0
# For u=0.6, icdf=1 since cdf(0)=0.504 <0.6, sum lt=1
print(m.entropy())  # ≈ tensor(1.1410)

tensor(0.6000)
tensor(0.4600)
tensor(0.8077)
tensor(0)
tensor([0., 0.])
tensor(-0.9213)
tensor(0.9020)
tensor(0)
tensor(0.9622)


In [3]:
probs = torch.tensor([0.1]*10 + [0.9]*10)  # n=20, to test dft
m = PoissonBinomial(probs=probs, method='dft')
print(m.mean)  # tensor(10.0000)  # 10*0.1 +10*0.9=1+9=10
print(m.variance)  # tensor(2.0000)  # 10*0.1*0.9 +10*0.9*0.1=10*0.09*2=1.8? Wait 20*0.09=1.8? No: 10*0.09 +10*0.09=1.8
print(m.skewness)  # tensor(0.0000)  # Symmetric
print(m.mode)  # tensor(10)
print(m.sample())  # e.g., tensor(10.)
print(m.log_prob(torch.tensor(10)))  # log PMF at 10, exact via DFT
print(m.cdf(torch.tensor(10)))  # cum up to 10
print(m.icdf(torch.tensor(0.5)))  # ≈ tensor(10)
# DFT should match DP for small n, but useful for larger.

tensor(10.)
tensor(1.8000)
tensor(-6.1704e-08)
tensor(11)
tensor(9.)
tensor(-1.5055)
tensor(0.3437)
tensor(11)


In [4]:
probs = torch.tensor([0.5] * 2000)  # n=2000, auto -> rna since >1024
m = PoissonBinomial(probs=probs)
print(m.mean)  # tensor(1000.)
print(m.variance)  # tensor(500.)  # 2000*0.5*0.5=500
print(m.skewness)  # tensor(0.0000)  # Since all p=0.5, (1-2p)=0
print(m.mode)  # tensor(1000)  # round mean
print(m.sample())  # tensor(1001.)  # etc., exact sample
print(m.log_prob(torch.tensor(1000)))  # Approx log(normal pdf-ish), ≈ log(1/sqrt(2 pi var)) ≈ -log(sqrt(2 pi 500)) ≈ -6.05
print(m.cdf(torch.tensor(1000)))  # ≈0.5  # normal.cdf(mean +0.5)
print(m.icdf(torch.tensor(0.5)))  # tensor(1000)  # Since gamma=0, same as normal
print(m.entropy())  # ≈ tensor(6.592)  # normal entropy = 0.5 log(2 pi e var) ≈ 0.5 log(2 pi e 500) ≈6.592

tensor(1000.)
tensor(500.)
tensor(0.)
tensor(1001)
tensor(990.)
tensor(-4.0274)
tensor(0.4911)
tensor(1001)
tensor(4.5295)


In [5]:
probs = torch.tensor([[0.1]*100, [0.9]*100])  # batch 2, n=100 each
m = PoissonBinomial(probs=probs, method='normal')
print(m.mean)  # tensor([10., 90.])
print(m.variance)  # tensor([9., 9.])  # 100*0.1*0.9=9
print(m.skewness)  # tensor([ 0.6667, -0.6667])  # (1-2p)>0 for p=0.1, negative for 0.9
print(m.log_prob(torch.tensor([10, 90])))  # Approx log P
print(m.cdf(torch.tensor([10, 90])))  # ≈ tensor([0.5, 0.5])
print(m.icdf(torch.tensor([0.95, 0.95])))  # Approx quantile, for normal ≈ mu + 1.645 sigma ≈10+1.645*3=14.9→15, 90+4.9→95

tensor([10.0000, 90.0000])
tensor([9.0000, 9.0000])
tensor([ 0.2667, -0.2667])
tensor([-2.0222, -2.0222])
tensor([0.5662, 0.5662])
tensor([15., 95.])


In [6]:
probs = torch.tensor([0.1]*100)  # n=100, mean=10, var=9, gamma≈0.6667 (positive skew)
m = PoissonBinomial(probs=probs, method='rna')
print(m.icdf(torch.tensor(0.95)))  # Adjusted: z=1.645, adj=1.645 + (1.645^2 -1)*0.6667/6 ≈1.645 + (2.706-1)*0.1111≈1.645+0.19≈1.835
# q=10 + 3*1.835≈10+5.5≈15.5→16 (higher than normal 15 due to positive skew)
# Other methods same as normal, but icdf adjusted.

tensor(14.)


In [7]:
# Small n: compare DP vs DFT
probs = torch.tensor([0.1, 0.2, 0.3])
m_dp = PoissonBinomial(probs=probs, method="dp")
m_dft = PoissonBinomial(probs=probs, method="dft")
print("DP PMF:", m_dp._log_pmf().exp())
print("DFT PMF:", m_dft._log_pmf().exp())

# Larger n: saddlepoint vs RNA
probs = torch.rand(2000)
m_auto = PoissonBinomial(probs=probs, method="auto")
print("Mean:", m_auto.mean.item())
print("Variance:", m_auto.variance.item())
print("Skewness:", m_auto.skewness.item())
print("CDF(1000):", m_auto.cdf(torch.tensor(1000.0)))
print("ICDF(0.95):", m_auto.icdf(torch.tensor(0.95)))


DP PMF: tensor([0.5040, 0.3980, 0.0920, 0.0060])
DFT PMF: tensor([0.5040, 0.0060, 0.0920, 0.3980])
Mean: 997.090576171875
Variance: 329.4726867675781
Skewness: 0.0002157005510525778
CDF(1000): tensor(0.4255)
ICDF(0.95): tensor(1034)
